<a href="https://colab.research.google.com/github/GustavoHenrr/projeto-RAG_JK/blob/main/notebooks/experimento_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
%cd /content
!rm -rf projeto-RAG_JK
!git clone https://github.com/GustavoHenrr/projeto-RAG_JK.git
%cd projeto-RAG_JK

/content
Cloning into 'projeto-RAG_JK'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 38 (delta 10), reused 30 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 17.02 MiB | 23.88 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/projeto-RAG_JK


In [12]:
!pip install -r requirements.txt

In [13]:
!grep -n "self_generate_response\|llm_instance.invoke\|def _gerar_resposta" rag_JK.py

121:    def _gerar_resposta(self, system_prompt: str, user_prompt: str) -> str:
127:        como _generate_response ou self_generate_response.
140:        response = self.llm_instance.invoke(chat_history, **self.params)


In [14]:
from rag_JK import JK_RAG

print("Classe JK_RAG importada com sucesso!")

Classe JK_RAG importada com sucesso!


In [15]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

local_llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 512,
        "do_sample": False,
        "return_full_text": False,
    },
)

chat_model = ChatHuggingFace(llm=local_llm)

print("Modelo carregado com sucesso!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado com sucesso!


In [16]:
rag = JK_RAG(
    llm_instance=chat_model,
    vector_db_path="vector_db",
    k=5
)

print("RAG instanciado com sucesso!")

Carregando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Carregando base vetorial em: /content/projeto-RAG_JK/vector_db
RAG carregado com sucesso!
RAG instanciado com sucesso!


In [17]:
pergunta = "Quem foi Juscelino Kubitschek?"

resposta = rag.answer_question(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta:")
print(resposta)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Pergunta:
Quem foi Juscelino Kubitschek?

Resposta:
Juscelino Kubitschek foi um político brasileiro que serviu como presidente do Brasil de 1956 a 1961. Ele é conhecido por ter implementado políticas de desenvolvimento econômico e social, incluindo a construção da cidade de Brasília.


In [ ]:
pergunta = "Quem foi Juscelino Kubitschek?"

documentos = rag._buscar_chunks_relevantes(pergunta)

for i, doc in enumerate(documentos, start=1):
    pagina = doc.metadata.get("page_number",  "página não informada")
    conteudo = doc.page_content.strip().replace("\n", " ")

    print("=" * 100)
    print(f"Trecho {i} | Página {pagina}")
    print("=" * 100)
    print(conteudo[:1000])
    print()

In [18]:
perguntas_experimento = [
    "Onde Juscelino Kubitschek nasceu?",
    "Quais informações o texto apresenta sobre a família de Juscelino Kubitschek?",
    "Como foi a formação acadêmica de Juscelino Kubitschek?",
    "Quais cargos políticos Juscelino Kubitschek ocupou?",
    "O texto menciona alguma relação de Juscelino Kubitschek com a medicina?"
]

valores_k = [3, 5, 10]

print("Perguntas e valores de k definidos com sucesso.")

Perguntas e valores de k definidos com sucesso.


In [19]:
def mostrar_chunks_recuperados(pergunta, valores_k, max_chars=500):
    """
    Mostra quais chunks foram recuperados para uma pergunta
    usando diferentes valores de k.
    """

    for k in valores_k:
        print("=" * 100)
        print(f"Valor de k: {k}")
        print(f"Pergunta: {pergunta}")
        print("=" * 100)

        rag.k = k
        documentos = rag._buscar_chunks_relevantes(pergunta)

        for i, doc in enumerate(documentos, start=1):
            pagina = doc.metadata.get("page_number", "página não informada")
            conteudo = doc.page_content.strip().replace("\n", " ")

            print(f"\nTrecho {i} | Página {pagina}")
            print(conteudo[:max_chars])
            print("-" * 100)


pergunta_teste_chunks = "Onde Juscelino Kubitschek nasceu?"

mostrar_chunks_recuperados(
    pergunta=pergunta_teste_chunks,
    valores_k=valores_k
)

Valor de k: 3
Pergunta: Onde Juscelino Kubitschek nasceu?

Trecho 1 | Página 239
JK • 257 Cronologia biográfica de JK 1830	 Chega ao Tijuco, depois Diamantina, o imigrante Jan Nepomuscky  Kubitschek, bisavô materno de Juscelino, marceneiro, descen­ dente de ciganos. Veio de Trebon, Tchecoslováquia, então parte  do império austro-húngaro. Casa-se com Teresa Maria de Jesus. 1872	 Nascimento do pai, João César de Oliveira, filho de Teófilo Gomes  de Oliveira e Eufrosina Leonardo Ribeiro. 1873	 Nascimento da mãe, Júlia Kubitschek, filha de Augusto Elias  Kubitschek e Maria Joaqu
----------------------------------------------------------------------------------------------------

Trecho 2 | Página 32
40 • JK A família de João César tinha forte tradição e muita história na região.  Já o primeiro Kubitschek de Diamantina, o imigrante Jan Nepomuscky  Kubitschek, bisavô materno de Juscelino, descendente de ciganos,  chegara à cidade em 1830. Viera de Trebon, Boêmia, Tchecoslováquia,  então part

In [20]:
import pandas as pd

resultados_experimento = []

for k in valores_k:
    print("=" * 100)
    print(f"Executando experimento com k = {k}")
    print("=" * 100)

    rag.k = k

    for pergunta in perguntas_experimento:
        print(f"Respondendo: {pergunta}")

        resposta = rag.answer_question(pergunta)

        resultados_experimento.append({
            "k": k,
            "pergunta": pergunta,
            "resposta": resposta
        })

df_resultados = pd.DataFrame(resultados_experimento)

df_resultados

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Executando experimento com k = 3
Respondendo: Onde Juscelino Kubitschek nasceu?
Respondendo: Quais informações o texto apresenta sobre a família de Juscelino Kubitschek?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respondendo: Como foi a formação acadêmica de Juscelino Kubitschek?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Respondendo: Quais cargos políticos Juscelino Kubitschek ocupou?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respondendo: O texto menciona alguma relação de Juscelino Kubitschek com a medicina?


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,k,pergunta,resposta
0,3,Onde Juscelino Kubitschek nasceu?,"Juscelino Kubitschek nasceu em Diamantina, est..."
1,3,Quais informações o texto apresenta sobre a fa...,O texto apresenta informações sobre a família ...
2,3,Como foi a formação acadêmica de Juscelino Kub...,Não encontrei informação suficiente no context...
3,3,Quais cargos políticos Juscelino Kubitschek oc...,Não encontrei informação suficiente no context...
4,3,O texto menciona alguma relação de Juscelino K...,"Sim, o texto menciona uma relação de Juscelino..."
5,5,Onde Juscelino Kubitschek nasceu?,Juscelino Kubitschek nasceu em Diamantina.
6,5,Quais informações o texto apresenta sobre a fa...,O texto apresenta algumas informações sobre a ...
7,5,Como foi a formação acadêmica de Juscelino Kub...,Não encontrei informação suficiente no context...
8,5,Quais cargos políticos Juscelino Kubitschek oc...,Não encontrei informação suficiente no context...
9,5,O texto menciona alguma relação de Juscelino K...,"Sim, o texto menciona várias relações de Jusce..."


In [21]:
for k in valores_k:
    print("=" * 100)
    print(f"RESULTADOS PARA k = {k}")
    print("=" * 100)

    resultados_k = df_resultados[df_resultados["k"] == k]

    for _, linha in resultados_k.iterrows():
        print("\nPergunta:")
        print(linha["pergunta"])

        print("\nResposta:")
        print(linha["resposta"])

        print("-" * 100)

RESULTADOS PARA k = 3

Pergunta:
Onde Juscelino Kubitschek nasceu?

Resposta:
Juscelino Kubitschek nasceu em Diamantina, estado de Minas Gerais, aos treze dias do mês de setembro do ano de 1902.
----------------------------------------------------------------------------------------------------

Pergunta:
Quais informações o texto apresenta sobre a família de Juscelino Kubitschek?

Resposta:
O texto apresenta informações sobre a família de Juscelino Kubitschek, incluindo:

- O bisavô materno, Jan Nepomuscky Kubitschek, imigrante tcheco que veio para Diamantina em 1830.
- O avô materno, Augusto Elias Kubitschek, que não concluiu os estudos e se casou jovem com Maria Joaquina Coelho.
- O pai de Juscelino, João César de Oliveira, que faleceu de tuberculose em 1915.
- A mãe de Juscelino, Júlia Kubitschek, filha de Augusto Elias Kubitschek e Maria Joaquina Coelho.
- Outras informações gerais sobre a família, como a tradição e a história familiar na região.
----------------------------------